# Hoja de Ejercicios 1

## Imports

In [16]:
from pathlib import Path
import os

from dotenv import load_dotenv
from Bio import SeqIO, Entrez
from Bio.SeqRecord import SeqRecord

## Configuración de Entrez
NCBI exige identificarse con un correo en cada consulta. Lo leemos una sola vez desde el archivo `.env` (variable `ENTREZ_EMAIL`) y se aplica a todos los ejercicios.

In [17]:
load_dotenv()

Entrez.email = os.getenv("ENTREZ_EMAIL")

## Ejercicio 1: Porcentaje GC

Escribe un programa que tome un fichero FASTA con varias secuencias de ADN y calcule el contenido de GC en porcentaje de cada una y lo muestre por pantalla.

A continuación produzca las cadenas de aminoácidos asociadas a cada secuencia de ADN.

Por último escriba estas secuencias de aminoácidos en un archivo FASTA utilizando como identificadores y descripciones las originales del archivo.

### 1. Rutas y carpeta de datos
Definimos la carpeta `data/ejercicio1` (se crea si no existe) y las rutas del FASTA de entrada y del FASTA de proteínas de salida.

In [18]:
ruta_datos = Path("data/ejercicio1")
ruta_datos.mkdir(parents=True, exist_ok=True)

ruta_entrada = ruta_datos / "secuencias_adn.fasta"
ruta_salida = ruta_datos / "proteinas.fasta"

### 2. Fichero FASTA de entrada
Descargamos desde la web del NCBI las secuencias codificantes (CDS) del plásmido ColE1 de *Escherichia coli* (`NC_001371`) en formato FASTA y las guardamos como `secuencias_adn.fasta`.

### 3. Porcentaje de GC, traducción y escritura del FASTA
Para cada secuencia del fichero:
1. Calculamos el porcentaje de GC: (G + C) / longitud × 100.
2. La traducimos a aminoácidos con `translate(table=11, cds=True)`: `table=11` es la tabla genética bacteriana (el plásmido es de *Escherichia coli*) y `cds=True` comprueba que la secuencia es una región codificante completa, convierte el codón de inicio alternativo (`GTG`) en `M` y elimina el codón de parada.
3. Guardamos la proteína con el identificador y la descripción originales.

Por último, escribimos todas las proteínas en `proteinas.fasta`.

In [19]:
proteinas = []

for registro in SeqIO.parse(ruta_entrada, "fasta"):
    secuencia = registro.seq.upper()

    porcentaje_gc = (secuencia.count("G") + secuencia.count("C")) / len(secuencia) * 100
    print(f"{registro.id}: {porcentaje_gc:.2f}% GC")

    proteina = secuencia.translate(table=11, cds=True)
    print(f"  Proteína: {proteina}\n")

    proteinas.append(SeqRecord(proteina, id=registro.id, description=registro.description))

numero = SeqIO.write(proteinas, ruta_salida, "fasta")
print(f"Se han escrito {numero} proteínas en {ruta_salida.as_posix()}")

lcl|NC_001371.1_cds_WP_000447006.1_1: 42.38% GC
  Proteína: METAVAYYKDGVPYDDKGQVIITLLNGTPDGSGSGGGGGKGGSKSESSAAIHATAKWSTAQLKKTQAEQAARAKAAAEAQAKAKANRDALTQRLKDIVNEALRHNASRTPSATELAHANNAAMQAEAERLRLAKAEEKARKEAEAAEKAFQEAEQRRKEIEREKAETERQLKLAEAEEKRLAALSEEAKAVEIAQKKLSAAQSEVVKMDGEIKTLNSRLSSSIHARDAEMKTLAGKRNELAQASAKYKELDELVKKLSPRANDPLQNRPFFEATRRRVGAGKIREEKQKQVTASETRINRINADITQIQKAISQVSNNRNAGIARVHEAEENLKKAQNNLLNSQIKDAVDATVSFYQTLTEKYGEKYSKMAQELADKSKGKKIGNVNEALAAFEKYKDVLNKKFSKADRDAIFNALASVKYDDWAKHLDQFAKYLKITGHVSFGYDVVSDILKIKDTGDWKPLFLTLEKKAADAGVSYVVALLFSLLAGTTLGIWGIAIVTGILCSYIDKNKLNTINEVLGI

lcl|NC_001371.1_cds_WP_010889865.1_2: 46.38% GC
  Proteína: MRKRFFVGIFAINLLVGCQANYIPDVQGGTIAPSSSSKLTGIAVQ

lcl|NC_001371.1_cds_WP_071531517.1_3: 57.86% GC
  Proteína: MHEPPVQPDRCALSGNYRLESNPVRHALTPLAAATGNRISRAMMAQTVLQSS

lcl|NC_001371.1_cds_WP_000165985.1_4: 52.08% GC
  Proteína: MTKQEKTALNMARFIRSQTLTLLEKLNELDADEQADICESLHDHADELYRSCLARFGDDGENL

lcl|NC_001371.1_cds_WP_000955999.1_5: 60.49% GC
  Proteína: MLTIRVTDDE

## Ejercicio 2: El sarampión

Escribe un *script* que automatice la descarga del genoma oficial del virus del sarampión.

1. Utiliza Entrez.esearch en la base de datos "nucleotide". Usa la base de datos **secundaria** y el término de búsqueda "Measles morbillivirus[Organism]"

2. Extrae el primer ID (IdList[0]) de los resultados devueltos.

3. Pásale ese ID a Entrez.efetch para descargar el registro en formato FASTA.

4. Imprime por pantalla la descripción del registro y los primeros 100 nucleótidos.

### 1. Búsqueda con `Entrez.esearch`
Buscamos en la base de datos `nucleotide` el término `Measles morbillivirus[Organism]`. Para usar la base de datos **secundaria** (RefSeq, curada) añadimos el filtro `srcdb_refseq[PROP]`.

In [20]:
termino = "Measles morbillivirus[Organism] AND srcdb_refseq[PROP]"
with Entrez.esearch(db="nucleotide", term=termino) as manejador:
    resultado = Entrez.read(manejador)

print(f"Resultados encontrados: {resultado['Count']}")

Resultados encontrados: 1


### 2. Primer ID de los resultados
`Entrez.read` devuelve un diccionario. Cogemos el primer identificador de la lista `IdList`.

In [21]:
id_registro = resultado["IdList"][0]
print(f"ID encontrado: {id_registro}")

ID encontrado: 9626945


### 3. Descarga con `Entrez.efetch`
Le pasamos el ID a `efetch` pidiendo el registro en formato FASTA y lo leemos con `SeqIO.read`, ya que solo hay un registro. Guardamos el registro en `data/ejercicio2/` para poder trabajar con él sin conexión.

In [22]:
ruta_datos = Path("data/ejercicio2")
ruta_datos.mkdir(parents=True, exist_ok=True)

with Entrez.efetch(db="nucleotide", id=id_registro, rettype="fasta", retmode="text") as manejador:
    registro = SeqIO.read(manejador, "fasta")

ruta_fasta = ruta_datos / f"{registro.id}.fasta"
SeqIO.write(registro, ruta_fasta, "fasta")

print(f"Registro descargado: {registro.id} ({len(registro.seq)} nucleótidos)")
print(f"Guardado en: {ruta_fasta.as_posix()}")

Registro descargado: NC_001498.1 (15894 nucleótidos)
Guardado en: data/ejercicio2/NC_001498.1.fasta


### 4. Descripción y primeros 100 nucleótidos
Mostramos la descripción del registro y los 100 primeros nucleótidos de la secuencia.

In [23]:
print(f"Descripción: {registro.description}")
print(f"Primeros 100 nucleótidos: {registro.seq[:100]}")

Descripción: NC_001498.1 Measles virus, complete genome
Primeros 100 nucleótidos: ACCAAACAAAGTTGGGTAAGGATAGATCAATCAATGATCATATTCTAGTACACTTAGGATTCAAGATCCTATTATCAGGGACAAGAGCAGGATTAGGGAT


## Ejercicio 3: La COVID-19

Tienes un archivo GenBank completo, pero solo te interesa saber cuántos genes tiene y cómo se llaman, ignorando el resto de secuencias reguladoras o estructurales.

Utilizando el genoma de referencia del SARS-CoV-2 (código de acceso NC_045512), escribe un programa que:

1. Descarga el archivo en formato GenBank (gb).

2. Recorre la tabla de características (features).

3. Filtra exclusivamente aquellas características cuyo tipo (.type) sea "gene".

4. Extrae el nombre del gen accediendo al diccionario de metadatos (qualifiers.get("gene")).

5. Muestra por pantalla un listado numerado con los nombres de todos los genes encontrados y sus coordenadas de inicio y fin.

### 1. Descarga con `Entrez.efetch`
Pedimos el registro `NC_045512` (genoma de referencia del SARS-CoV-2) a la base de datos `nucleotide` en formato **GenBank** (`gb`) y lo leemos con `SeqIO.read`, ya que solo hay un registro. Guardamos el registro en `data/ejercicio3/` para poder trabajar con él sin conexión.

In [24]:
ruta_datos = Path("data/ejercicio3")
ruta_datos.mkdir(parents=True, exist_ok=True)

with Entrez.efetch(db="nucleotide", id="NC_045512", rettype="gb", retmode="text") as manejador:
    registro = SeqIO.read(manejador, "genbank")

ruta_gb = ruta_datos / f"{registro.id}.gb"
SeqIO.write(registro, ruta_gb, "genbank")

print(f"Registro descargado: {registro.id} ({len(registro.seq)} nucleótidos)")
print(f"Guardado en: {ruta_gb.as_posix()}")

Registro descargado: NC_045512.2 (29903 nucleótidos)
Guardado en: data/ejercicio3/NC_045512.2.gb


### 2. Recorrer las características y filtrar los genes
`registro.features` es la tabla de características del registro (genes, CDS, regiones UTR...). Recorremos la tabla y nos quedamos solo con las de tipo `"gene"`.

In [25]:
genes = []

for caracteristica in registro.features:
    if caracteristica.type == "gene":
        genes.append(caracteristica)

print(f"Características de tipo gene: {len(genes)}")

Características de tipo gene: 11


### 3. Nombre y coordenadas de cada gen
El nombre está en el diccionario `qualifiers` (con `.get("gene")`, que devuelve una lista, por eso el `[0]`). Las coordenadas están en `location`. Biopython cuenta desde 0, así que sumamos 1 al inicio para mostrarlas como en GenBank.

In [26]:
for numero, gen in enumerate(genes, start=1):
    nombre = gen.qualifiers.get("gene")[0]
    inicio = gen.location.start + 1
    fin = gen.location.end
    print(f"{numero}. {nombre}: {inicio} - {fin}")

1. ORF1ab: 266 - 21555
2. S: 21563 - 25384
3. ORF3a: 25393 - 26220
4. E: 26245 - 26472
5. M: 26523 - 27191
6. ORF6: 27202 - 27387
7. ORF7a: 27394 - 27759
8. ORF7b: 27756 - 27887
9. ORF8: 27894 - 28259
10. N: 28274 - 29533
11. ORF10: 29558 - 29674


## Ejercicio 4: Bacterias: A la caza de proteínas de resistencia

Las bacterias tienen un “superpoder”: Pueden pasarse pequeños anillos de AND circular llamados plásmidos entre ellas. Estos plásmidos suelen contener genes que codifican proteinas de resistencia a los antibióticos.

Para averiguar contra qué antibióticos se está defendiendo una bacteria mutirresistente el primer paso es **secuenciar** el plásmido. En nuestro caso descargaremos la secuencia de su plásmido, aislaremos los genes codificantes (CDS) y los traduciremos a proteínas.

---

Escribe un script que automatice la caza de proteínas de resistencia.

Utilizaremos el plásmido pT181 del patógeno Staphylococcus aureus, cuyo código de acceso en el NCBI es NC_001371.

El programa debe conectarse al NCBI, descargar el plásmido en formato GenBank (gb), buscar todas las regiones CDS y traducirlas a proteínas.

### 1. Descarga del plásmido en formato GenBank
Descargamos con `Entrez.efetch` el plásmido pT181 de *Staphylococcus aureus* (`NC_001393`) en formato `gb` y lo leemos con `SeqIO.read`, ya que solo hay un registro. El enunciado indica `NC_001371`, pero ese código corresponde en el NCBI al plásmido ColE1 de *Escherichia coli*, que no contiene genes de resistencia a antibióticos; por eso usamos el registro real de pT181. Usamos `rettype="gbwithparts"` (variante de GenBank) porque con `gb` el registro llega sin la secuencia y no podríamos extraer ni traducir los CDS. Guardamos el registro en `data/ejercicio4/` para poder trabajar con él sin conexión.

In [27]:
ruta_datos = Path("data/ejercicio4")
ruta_datos.mkdir(parents=True, exist_ok=True)

with Entrez.efetch(db="nucleotide", id="NC_001393", rettype="gbwithparts", retmode="text") as manejador:
    plasmido = SeqIO.read(manejador, "genbank")

ruta_gb = ruta_datos / f"{plasmido.id}.gb"
SeqIO.write(plasmido, ruta_gb, "genbank")

print(f"Plásmido descargado: {plasmido.id} ({len(plasmido.seq)} pb)")
print(f"Primeros 30 nucleótidos: {plasmido.seq[:30]}")
print(f"Guardado en: {ruta_gb.as_posix()}")

Plásmido descargado: NC_001393.1 (4439 pb)
Primeros 30 nucleótidos: TTGGAAATCTCAACCGAGACAACGCTCAAG
Guardado en: data/ejercicio4/NC_001393.1.gb


### 2. Búsqueda de las regiones CDS
Recorremos la tabla de características (`features`) y nos quedamos solo con las de tipo `CDS`, que son las regiones codificantes.

In [28]:
cds = []

for caracteristica in plasmido.features:
    if caracteristica.type == "CDS":
        cds.append(caracteristica)

print(f"CDS encontrados: {len(cds)}")

CDS encontrados: 3


### 3. Traducción a proteínas
Para cada CDS extraemos su secuencia de ADN (`extract` tiene en cuenta la hebra y las coordenadas) y la traducimos con la tabla genética bacteriana (11). Si el CDS es completo, usamos `cds=True` para comprobarlo y eliminar el codón de parada. Mostramos el nombre del producto y su proteína.

El CDS de la bomba de expulsión de tetraciclina Tet(K), la proteína de resistencia del plásmido, está marcado en RefSeq como pseudogén (`/pseudo`, "frameshifted"). Como es justo la proteína que buscamos, la traducimos igualmente, sin `cds=True` (que daría error) y avisando de que la traducción contiene codones de parada internos.

In [29]:
for numero, caracteristica in enumerate(cds, start=1):
    producto = caracteristica.qualifiers.get("product", ["desconocido"])[0]
    secuencia_cds = caracteristica.extract(plasmido.seq)

    if "pseudo" in caracteristica.qualifiers:
        proteina = secuencia_cds.translate(table=11)
        print(f"{numero}. {producto} ({len(proteina)} aa) [PSEUDOGÉN]")
        print(f"   Aviso: RefSeq lo marca como pseudogén (marco de lectura desplazado); la traducción contiene {proteina.count('*')} codones de parada internos (*).")
    else:
        proteina = secuencia_cds.translate(table=11, cds=True)
        print(f"{numero}. {producto} ({len(proteina)} aa)")

    print(f"   {proteina}\n")

1. replication initiation factor domain-containing protein (314 aa)
   MYKNNHANHSNHLENHDLDNFSKTGYSNSRLDAHTVCISDPKLSFDAMTIVGNLNRDNAQALSKFMSVEPQIRLWDILQTKFKAKALQEKVYIEYDKVKADSWDRRNMRIEFNPNKLTRDEMIWLKQNIISYMEDDGFTRLDLAFDFEEDLSDYYAMSDKAVKKTIFYGRNGKPETKYFGVRDSNRFIRIYNKKQERKDNADAEVMSEHLWRVEIELKRDMVDYWNDCFSDLHILQPDWKTIQRTADRAIVFMLLSDEEEWGKLHRNSRTKYKNLIKEISPVDLTDLMKSTLKANEKQLQKQIDFWQHEFKFWK

2. tetracycline efflux MFS transporter Tet(K) (460 aa) [PSEUDOGÉN]
   Aviso: RefSeq lo marca como pseudogén (marco de lectura desplazado); la traducción contiene 15 codones de parada internos (*).
   LFSLYKKFKGLFYSVLFWLCILSFFSVLNEMVLNVSLPDIANHFNTTPGITNWVNTAYMLTFSIGTAVYGKLSDYINIKKLLIIGISLSCLGSLIAFIGHNHFFILIFGRLVQGVGSAAFPSLIMVVVARNITRKKQGKAFGFIGSIVALGEGLGPSIGGIIAHYIHWSYLLILPMITIVTIPFLIKVMVPGKSTKNTLDIVGIVLMSISIICFMLFTTNYNWTFLILFTIFFVIFIKHISRVSNPFINPKLGKNIPFMLGLFSGWANIFYSSWFYINGALYDENYLSCKCSDNR**CYFSWNHECYCFWLFWWFFSG*KRIIICFYFRIIVYLYKFFNYCIFC*V*YVVDYFYVYICYGRIIFY*NSYIKNSIK*SF*RRSCFWNEFAKFHKFFIRGNRYSNCRRFIVTTID

## Ejercicio 5: Algunas proteínas "célebres"

Buscar información, y archivos FASTA asociados, relativa a tres de estas proteínas.

| Hemoglobina | Colágeno | Queratina | Enzima amilasa | Tubulina |
|---|---|---|---|---|
| Mioglobina | Albúmina | Fibrinógeno | Fibrina | Ferritina |
| Zeina | Elastina | Gluteína | Pepsina | Trombina |
| Prolactina | Actina | Miosina | Caseína | Calcitonina |
| Oxitocina | Gammaglobulina | Enzima proteasa | Enzima lipasa | Inmunoglobulina |
| Colágeno | Queratina | Albúmina | Glucagón | Hipocretina |



### 1. Elección de las tres proteínas
Elegimos tres proteínas de la tabla y buscamos su código en UniProt, una base de datos de proteínas (todas son humanas):

| Proteína | Código |
|---|---|
| Mioglobina | `P02144` |
| Albúmina | `P02768` |
| Prolactina | `P01236` |

### 2. Descarga de los archivos FASTA
Descargamos el FASTA de cada proteína con `Entrez.efetch` y lo guardamos en `data/ejercicio5/`. Para cada una mostramos su nombre, longitud y los primeros aminoácidos.

In [30]:
proteinas = {
    "Mioglobina": "P02144",
    "Albúmina": "P02768",
    "Prolactina": "P01236",
}

ruta_datos = Path("data/ejercicio5")
ruta_datos.mkdir(parents=True, exist_ok=True)

for nombre, codigo in proteinas.items():
    with Entrez.efetch(db="protein", id=codigo, rettype="fasta", retmode="text") as manejador:
        registro = SeqIO.read(manejador, "fasta")

    ruta_fasta = ruta_datos / f"{codigo}.fasta"
    SeqIO.write(registro, ruta_fasta, "fasta")

    print(nombre)
    print(f"  Identificador: {registro.id}")
    print(f"  Descripción: {registro.description}")
    print(f"  Longitud: {len(registro.seq)} aminoácidos")
    print(f"  Primeros 30 aminoácidos: {registro.seq[:30]}")
    print(f"  Guardado en: {ruta_fasta.as_posix()}\n")

Mioglobina
  Identificador: sp|P02144.2|MYG_HUMAN
  Descripción: sp|P02144.2|MYG_HUMAN RecName: Full=Myoglobin; AltName: Full=Nitrite reductase MB; AltName: Full=Pseudoperoxidase MB
  Longitud: 154 aminoácidos
  Primeros 30 aminoácidos: MGLSDGEWQLVLNVWGKVEADIPGHGQEVL
  Guardado en: data/ejercicio5/P02144.fasta

Albúmina
  Identificador: sp|P02768.2|ALBU_HUMAN
  Descripción: sp|P02768.2|ALBU_HUMAN RecName: Full=Albumin; Flags: Precursor
  Longitud: 609 aminoácidos
  Primeros 30 aminoácidos: MKWVTFISLLFLFSSAYSRGVFRRDAHKSE
  Guardado en: data/ejercicio5/P02768.fasta

Prolactina
  Identificador: sp|P01236.1|PRL_HUMAN
  Descripción: sp|P01236.1|PRL_HUMAN RecName: Full=Prolactin; Short=PRL; Flags: Precursor
  Longitud: 227 aminoácidos
  Primeros 30 aminoácidos: MNIKGSPWKGSLLLLLVSNLLLCQSVAPLP
  Guardado en: data/ejercicio5/P01236.fasta



### 3. Información de cada proteína

| Proteína | Gen | Dónde está | Qué hace | Dato concreto |
|---|---|---|---|---|
| Mioglobina | `MB` | Músculos (corazón y esqueléticos) | Guarda oxígeno y lo cede cuando el músculo hace esfuerzo. | Fue la primera proteína cuya forma en 3D se descubrió (John Kendrew, 1958). Es la que da color rojo a la carne. |
| Albúmina | `ALB` | Se produce en el hígado y circula por la sangre | Transporta hormonas, grasas y medicamentos, y evita que el agua salga de la sangre a los tejidos. | Es la proteína más abundante del plasma (entre 35 y 50 g por litro de sangre). Un nivel bajo puede indicar problemas de hígado o riñón. |
| Prolactina | `PRL` | Se produce en la hipófisis, una glándula del cerebro | Es una hormona que estimula la producción de leche tras el parto. | Sus niveles suben durante el embarazo y la lactancia. |

Fuentes: fichas de UniProt de [Mioglobina](https://www.uniprot.org/uniprotkb/P02144/entry), [Albúmina](https://www.uniprot.org/uniprotkb/P02768/entry) y [Prolactina](https://www.uniprot.org/uniprotkb/P01236/entry).